# Path 3 — strategy screen (Nov 5, 2026)

Four simple strategies (S0a greedy, S0b″ gated noise + hold, S2 stuck timer + yield + hold, S1′ amplified averaging)
simulated on the lab v09 interface (`src/slider/path3/`). Runs come from `scripts/path3_screen.py` → `results/path3/runs.csv`.
Human inputs (reaction / start delays, consensus times) are lab v09 G-train rounds only; no test runs are used.
The graph split is `data/splits/graph_split.csv`; graph-test is never used.

In [ ]:
import json

import numpy as np
import pandas as pd

from slider import config
from slider.path3 import graphs, human, report

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
runs = pd.read_csv(config.PATH3 / "runs.csv")
screen = runs[runs["block"].isin(["screen_A", "screen_B"])]
sweeps = runs[runs["block"].isin(["sweep_A", "sweep_B"])]
raw = runs[runs["block"].isin(["raw_A", "raw_B"])]
order = list(json.loads((config.PATH3 / "screening_set.json").read_text()))
runs.groupby(["block", "strategy"]).size().unstack()

## Screening set: 5 human networks + 2 graph-train graphs per (fraught tercile × |λ| tercile) cell

In [ ]:
split = graphs.load_split()
split[split["graph"].isin(order) | split["human"].notna()][["graph", "human", "n_edges", "n_fraught", "lam", "cell", "split"]]

## Rankings (per graph and pooled over the screening set; Agreement read as integer %)

Rank: success_300 (high first), then median time (a run without consensus counts as infinite).

In [ ]:
rank = report.rankings(screen)
rank.to_csv(config.PATH3 / "rankings.csv", index=False)
rank[rank["graph"] == "pooled"].round(3)

In [ ]:
rank[rank["graph"].isin(["Hierarchy", "Full", "Ring", "Star", "CT"])].round(3)

## Sensitivity: Agreement read unrounded (pooled)

In [ ]:
report.rankings(raw).query("graph == 'pooled'").round(3)

## Efficiency gap: human median consensus time ÷ strategy median time (start A)

Only Hierarchy and Full have lab v09 G-train rounds (Star has none in lab v09; Ring and CT are test topologies).

In [ ]:
gap = report.efficiency_gap(screen[screen["start"] == "A"], pd.read_csv(human.CONSENSUS))
gap.round(2)

## Figure 1: Hierarchy sweeps (median time vs ε, H, T, p, κ)

In [ ]:
report.sweep_figure(sweeps, config.FIGURES / "path3_sweeps.png");

## Figure 2: strategy × screening graph, median time (start A)

In [ ]:
report.heatmap_figure(screen[screen["start"] == "A"], order, config.FIGURES / "path3_heatmap.png");